# **Task 4 — Distributed Computing**

In [1]:
import time
import pandas as pd

from google.colab import drive

from pyspark.sql import SparkSession

from pyspark import StorageLevel

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
spark = (
    SparkSession.builder
    .appName("Task4_Distributed_Computing")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

SEED = 42

In [4]:
processed_df = spark.read.parquet(
    "/content/drive/MyDrive/7006SCN/processed_reviews"
)

In [5]:
print("="*60)
print("Spark Resource Configuration")
print("="*60)

print("Driver Memory:",
      spark.conf.get("spark.driver.memory"))

print("Executor Memory:",
      spark.conf.get("spark.executor.memory"))

print("Shuffle Partitions:",
      spark.conf.get("spark.sql.shuffle.partitions"))

Spark Resource Configuration
Driver Memory: 4g
Executor Memory: 4g
Shuffle Partitions: 8


In [6]:
print("Current Partitions")

print(processed_df.rdd.getNumPartitions())

Current Partitions
3


In [7]:
print("Benchmark Without Cache")

start = time.time()

processed_df.groupBy("label").count().show()

end = time.time()

without_cache = end-start

print(f"Execution Time: {without_cache:.2f} seconds")

Benchmark Without Cache
+-----+------+
|label| count|
+-----+------+
|  4.0| 24669|
|  1.0| 72795|
|  0.0|304578|
|  3.0| 37256|
|  2.0| 42006|
+-----+------+

Execution Time: 9.29 seconds


In [8]:
print("Caching Dataset")

processed_df.cache()

processed_df.count()

print("Dataset Cached Successfully")

Caching Dataset
Dataset Cached Successfully


In [9]:
print("Benchmark Cached Dataset")

start = time.time()

processed_df.groupBy("label").count().show()

end = time.time()

cache_time = end-start

print(f"Execution Time: {cache_time:.2f} seconds")

Benchmark Cached Dataset
+-----+------+
|label| count|
+-----+------+
|  4.0| 24669|
|  1.0| 72795|
|  0.0|304578|
|  3.0| 37256|
|  2.0| 42006|
+-----+------+

Execution Time: 1.01 seconds


In [10]:
processed_df.persist(StorageLevel.MEMORY_AND_DISK)

processed_df.count()

print("Dataset Persisted Successfully")

Dataset Persisted Successfully


In [11]:
start = time.time()

processed_df.groupBy("label").count().show()

end = time.time()

persist_time = end-start

print(f"Execution Time: {persist_time:.2f} seconds")

+-----+------+
|label| count|
+-----+------+
|  4.0| 24669|
|  1.0| 72795|
|  0.0|304578|
|  3.0| 37256|
|  2.0| 42006|
+-----+------+

Execution Time: 0.54 seconds


In [12]:
print("Before Repartition")

print(processed_df.rdd.getNumPartitions())

repartition_df = processed_df.repartition(16)

print("After Repartition")

print(repartition_df.rdd.getNumPartitions())

Before Repartition
3
After Repartition
16


In [13]:
start = time.time()

repartition_df.groupBy("label").count().show()

end = time.time()

repartition_time = end-start

print(f"Execution Time: {repartition_time:.2f} seconds")

+-----+------+
|label| count|
+-----+------+
|  4.0| 24669|
|  1.0| 72795|
|  0.0|304578|
|  3.0| 37256|
|  2.0| 42006|
+-----+------+

Execution Time: 1.60 seconds


### **Performance Comparison**

In [14]:
performance = pd.DataFrame({

    "Method":[
        "Without Cache",
        "Cache",
        "Persist",
        "Repartition"
    ],

    "Execution Time":[
        without_cache,
        cache_time,
        persist_time,
        repartition_time
    ]

})

performance

,Method,Execution Time
0,Without Cache,9.290040
1,Cache,1.007883
2,Persist,0.538524
3,Repartition,1.597242


In [15]:
print(processed_df.storageLevel)

Disk Memory Deserialized 1x Replicated


In [16]:
performance.to_csv(

    "/content/drive/MyDrive/7006SCN/performance_summary.csv",

    index=False

)

print("Performance summary saved successfully.")

Performance summary saved successfully.


In [17]:
spark.sparkContext.getConf().getAll()

[('spark.rdd.compress', 'True'),
 ('spark.driver.port', '44877'),
 ('spark.hadoop.fs.s3a.vectored.read.min.seek.size', '128K'),
 ('spark.app.name', 'Task4_Distributed_Computing'),
 ('spark.executor.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-